In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from torch.optim.lr_scheduler import PolynomialLR

torch.manual_seed(42)

### План 

**Нормализующие потоки**

1. Одномерное распределение: смесь гауссов.
2. Двумерное распределение (изображение).

**Обратные задачи**

1. Задача Кеплера (?): восстановление параметров массы/энергии (?)

## Нормализующие потоки


- обратимые архитектуры с вычислимым якобианом:

$$
\text{NF} = g() \colon  z = g(y), \ \ y = g^{-1}(z)\equiv f(z)
$$



Позволяют сэмплировать из сложного распределения $p_Y$ за счет преобразования нормально распределенной $k$-мерной случайной величины $p_Z \sim \mathcal{N}(0, I_k)$.

В явном виде:
$$
p_Y(y) = p_Z(g(y))\left| \det \frac{\partial g}{\partial y}\right| = p_Z(z) \left| \det \frac{\partial f}{\partial z}\right|^{-1}
$$




`zuko` - фреймворк для работы с нормализующими потоками (использует `torch`): 

**Документация:**

https://zuko.readthedocs.io/0.1.5/api/zuko.flows.html

**Исходный код:**

https://github.com/probabilists/zuko/tree/89aef8c602e7a4bb173ad8404551372c9eafb6da

In [1]:
#!pip install zuko

In [7]:
import zuko

## 1D: смесь гауссов

### Задание 1

Сгенерируйте тренировочные данные: $y\sim \sum_i \mu_i N(\mu_i, \sigma_i), \ i\in [1, 4]$ на основе выборок и pdf. Сравните с $N(0, 1)$. 

In [3]:
def sample_target(n_samples):
    """
    Generate samples from a 4-Gaussian mixture
    """
    means = [-1.5, -0.5, 0.5, 1.5]  
    sigmas = [0.4, 0.3, 0.4, 0.35] 
    amps = [0.2, 0.3, 0.1, 0.4]
    
    # Select components randomly
    components = ...

    # Generate samples
    samples_norm = np.zeros(n_samples)

    for i, (mu, sigma) in enumerate(zip(means, sigmas)):
        ...

    return samples_norm.astype(np.float32)


def target_pdf(x):
    """PDF for 4-Gaussian mixture"""
    
    means = [-1.5, -0.5, 0.5, 1.5]  
    sigmas = [0.4, 0.3, 0.4, 0.35] 
    amps = [0.2, 0.3, 0.1, 0.4]
    
    x = np.asarray(x)
    pdf = np.zeros_like(x)

    for mu, sigma, amp in zip(means, sigmas, amps):
        ...

    return pdf

In [ ]:
# TODO: generate samples and calculate pdf on grid. Plot the result 

target_samples = ... # N = 10000

y_grid = np.linspace(-5, 5, 500)
pdf_target = ... 

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(y_grid, pdf_target, 'b-', linewidth=2, label='Target (4 Gaussians)')
axes[0].set_xlabel('x', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].set_title('', fontsize=14)
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].hist(target_samples, bins=80, density=True, alpha=0.5, label='Target', range=(-3,3))

axes[1].set_xlabel('x', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('Sample Comparison', fontsize=14)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


### Задание 2

1. Изучите класс NormalizingFlow 
    - **Исходный код:** https://github.com/probabilists/zuko/blob/89aef8c602e7a4bb173ad8404551372c9eafb6da/zuko/distributions.py

    - `self.base`?
    - метод `log_prob()`?
    - метод `rsample`?
    
    - `self.transform`: `MonotonicRQSTransform`. **Исходный код:** 
    https://github.com/probabilists/zuko/blob/89aef8c602e7a4bb173ad8404551372c9eafb6da/zuko/transforms.py


2. Инициализируйте 
    - модель **Neural Spline Flow** для решения задачи, а также 
    - оптимизатор (Adam), 
    - lr_scheduler (see: https://docs.pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.PolynomialLR.html)


In [10]:
n_dim = ...
hidden_shape = [256, 256]
layers = 14

nf_model =  ...
optimizer = ...

In [11]:

max_epochs = 14_000
scheduler = ...

batch_size = 512
losses = []


### Задание 3

1. Реализуйте процедуру обучения. 

    <details> 
    <summary>Вспомните, как обучать NF? </summary>
    Максимизировать правдоподобие: хотим, чтобы $p_Y(D/\theta)$ было максимальным

    $$
    L = - \log p(D|\theta) = ... ?
    $$
    </details>


In [ ]:

print("Training")
for epoch in range(max_epochs):
    optimizer.zero_grad()
        
    # Sample from target
     
    y = ... 

    # Compute log probability under the flow
    log_prob = ... 

    loss = -log_prob.mean()
    
    loss.backward()
    optimizer.step()
    scheduler.step()
    
    losses.append(loss.item())
    
    if (epoch + 1) % 500 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{max_epochs}, Loss: {loss.item():.4f}, LR: {current_lr:.2e}")


2. Оцените качество вашей модели. Можно использовать `from scipy.stats import gaussian_kde` для оценки pdf


In [16]:

nf_model.eval()
with torch.no_grad():
    y_nf_samples = nf_model().sample((10000,)).numpy().flatten()


target_samples = ...

y_grid = np.linspace(-5, 5, 500)
pdf_target = target_pdf(y_grid)



In [17]:
from scipy.stats import gaussian_kde

kde_flow = gaussian_kde(y_nf_samples)
pdf_flow = kde_flow(y_grid)

In [ ]:


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(y_grid, pdf_target, 'b-', linewidth=2, label='Target (4 Gaussians)')
axes[0].plot(y_grid, pdf_flow, 'r--', linewidth=2, label='MAF')
axes[0].set_xlabel('x', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].set_title('', fontsize=14)
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].hist(target_samples, bins=80, density=True, alpha=0.5, label='Target', range=(-3,3))
axes[1].hist(y_nf_samples, bins=80, density=True, alpha=0.5, label='MAF', range=(-3,3))
axes[1].set_xlabel('x', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('Sample Comparison', fontsize=14)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


KL divergence на основе pdf:

In [ ]:
eps = 1e-10
kl_div = np.sum(pdf_target * np.log((pdf_target+eps)/(pdf_flow+eps))) * (y_grid[1]-y_grid[0])
print(f"\nKL Divergence: {kl_div:.4f}")

3. Получите распределение $p_Y$ в явном виде

In [2]:
y = torch.linspace(-4, 4, 500).reshape(-1, 1)

with torch.no_grad():
    log_p_z_plus_log_det_j = ...  # log_det = log|det df/dz|

# 3. p(y) = p(z) * exp(log_det)
p_y = ...

In [21]:
with torch.no_grad():
    z, _ = nf_model().transform.call_and_ladj(y) # get z from y 
    p_z = ... # use .base.log_prob(z)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(y, p_y, 'b-', linewidth=2, label='Target (4 Gaussians)')
axes[1].plot(z, p_z, 'r--', linewidth=2, label='Base distribution')
axes[0].set_xlabel('x', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].set_title('', fontsize=14)
axes[0].legend()
axes[1].legend()
axes[0].grid(alpha=0.3)

## 2D: генерация изображения

In [194]:
from utils import generate_2d_data

### Задание 1.

1. Сгенерируйте 2d распределение (тренировочный набор)
2. Постройте модель на основе NSF для данной задачи
3. Реализуйте процедуру обучения (аналогично случаю 1D)

In [ ]:
xy = generate_2d_data('rings')[0]

fig = plt.figure(figsize=(6, 5))
plt.scatter(xy[:, 0], xy[:, 1], s=3, color='b')


In [41]:
def plot_heatmap_2d(dist, xmin=-4.0, xmax=4.0, ymin=-4.0, ymax=4.0, mesh_count=1000, name=None):
    ## Plot proba obtained with NF as a heatmap

    plt.figure()
    
    x = torch.linspace(xmin, xmax, mesh_count)
    y = torch.linspace(ymin, ymax, mesh_count)
    X, Y = torch.meshgrid(x, y)
    
    concatenated_mesh_coordinates = torch.stack([Y.reshape(-1), X.reshape(-1)]).t()
    prob = ... # get log_prob for each point on your mesh grid 
    prob = prob.numpy()
    
    plt.title(name)
    plt.imshow(np.reshape(prob, (mesh_count, mesh_count)).T, origin="lower")
    plt.xticks([0, mesh_count * 0.25, mesh_count * 0.5, mesh_count * 0.75, mesh_count], [xmin, xmin/2, 0, xmax/2, xmax])
    plt.yticks([0, mesh_count * 0.25, mesh_count * 0.5, mesh_count * 0.75, mesh_count], [ymin, ymin/2, 0, ymax/2, ymax])
    plt.show()

In [3]:
n_dim = ... 
hidden_shape = [256, 256]
layers = 14

nf_2d_model = ... 
optimizer = ...

In [4]:
max_epochs = 4_000
base_lr = 1e-3

scheduler = ...

batch_size = 512
losses = []


In [ ]:

print("Training")
for epoch in range(max_epochs):
    ...
    
    losses.append(loss.item())
    
    if (epoch) % 500 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{max_epochs}, Loss: {loss.item():.4f}, LR: {current_lr:.2e}")
        with torch.no_grad():
            plot_heatmap_2d(nf_2d_model, -4.0, 4.0, -4.0, 4.0, mesh_count=200, name=str(epoch))

In [ ]:
with torch.no_grad():
    plot_heatmap_2d(nf_2d_model, -4.0, 4.0, -4.0, 4.0, mesh_count=200, name='final')

## Обратная задача Кеплера


$$
x=a(\cos E − e),\ \ y= a \sqrt{1−e^2}
​$$

Временная динамика 

$$
M=E−e\sin E
$$

$M=2\pi(t−t_0​)/T$ − средняя аномалия, $E$ - эксцентрисетическая аномалия. 


In [43]:
import FrEIA.framework as Ff
import FrEIA.modules as Fm

### Задание 1 

1. Сгенерируйте набор орбит с различными параметрами e, M (тренировочный набор)
2. Визуализируйте несколько примеров

In [158]:
def generate_orbit(e, M, n_points=50, noise_std=None):
    M_vals = np.linspace(0, 2*np.pi, n_points) + M
    xy_points = []

    for M_k in M_vals:
        E_anom = M_k
        for _ in range(8):
            E_anom -= (E_anom - e*np.sin(E_anom) - M_k) / (1 - e*np.cos(E_anom))
        f = 2 * np.arctan2(np.sqrt(1+e)*np.sin(E_anom/2),
                           np.sqrt(1-e)*np.cos(E_anom/2))
        r = 1 - e*np.cos(E_anom)
        x, y = r*np.cos(f), r*np.sin(f)
        xy_points.append([x, y])

    xy_points = np.array(xy_points)

    if noise_std is not None and noise_std > 0:
        xy_points += np.random.normal(0, noise_std, size=xy_points.shape)

    return xy_points.astype(np.float32) 


n_samples = 5000
n_points = 100

orbits = [] 
params = [] # Physical parameters: [e, M]

for _ in range(n_samples):
    e =  ...    # Eccentricity
    M =  ...   # Mean anomaly (start point in radians)
    orbit = ...  # use noise 0.02
    
    orbits.append(orbit)
    params.append([e, M])

orbits_t = torch.tensor(np.array(orbits), dtype=torch.float32)
params_t =  torch.tensor(np.array(params), dtype= torch.float32)

In [195]:
from utils import plot_kepler

In [ ]:
# checking 
for j in [100, 939, 3928]:

    xy = orbits_t[j].squeeze()
    p = params_t[j].numpy()

    true_orbit = ... # orbit without noise

    true_orbit = torch.tensor(true_orbit)
    
    print('Number of orbit', j, f'\n e, M = {p}')
    plot_kepler(true_orbit[:, 0], true_orbit[:, 1], xy[:, 0], xy[:, 1])

### Задание 2

1. Реализуйте простой трехслойный кодировщик, чтобы сжимать орбиту до сжатого представления


In [ ]:
# ! pip install FrEIA

In [ ]:
import FrEIA.framework as Ff
import FrEIA.modules as Fm

In [115]:
class OrbitEncoder(nn.Module):
    def __init__(self, input_dim=..., latent_dim=...):
        super().__init__()
        self.encoder = nn.Sequential(
            ...
        )
    
    def forward(self, orbits):
        return ...

2. Реализуйте `cINN`
    - коэффициенты в обратимом блоке задаются с помощю полносвязной модели subnet_fc
    - нужно правильно задать все размерности 
    - какая размерность будет у параметров $z$?


In [ ]:
def subnet_fc(c_in, c_out):
    return nn.Sequential(
       ... # use e.g 512 hidden layer size
    )

cinn = Ff.SequenceINN(...) # How many output parameters?

for k in range(12):
    cinn.append(
        Fm.AllInOneBlock,
        cond=0,
        cond_shape=(... ,), # What is the shape of condition?
        subnet_constructor=subnet_fc
    )

print(cinn)


In [ ]:

encoder = ...
optimizer = torch.optim.Adam(list(cinn.parameters()) + list(encoder.parameters()), lr=1e-3)


3. Реализуйте процедуру обучения
    - условие?
    - z = cinn(y, condition)
    - loss? максимальное правдоподобие (с обратным знаком)

In [ ]:
flat_orbits = ... # make orbits flat. why?

for epoch in range(100):
    condition =  ... 
    z, log_jac = ...
    loss = ... 
    ... 
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.3f}")
 

In [ ]:
cinn.eval()
encoder.eval()

def infer_params(orbit, n_samples_z=1000):
    """
    orbit: tensor of shape (n_points, 2)
    returns: posterior samples of (e, M), and their mean
    """
    with torch.no_grad():
        cond = encoder(orbit.view(1, -1))          # [1, 2]
        cond = cond.repeat(n_samples_z, 1)         # [K, 2]

        # latent prior samples
        z = torch.randn(n_samples_z, 2)

        # reverse pass: z -> params | cond
        params_samples, _ = cinn(z, c=[cond], rev=True)
        e_samples = params_samples[:, 0]
        M_samples = params_samples[:, 1]

        e_mean = e_samples.mean()
        M_mean = M_samples.mean()

    return params_samples, e_mean.item(), M_mean.item()

orbit = torch.tensor(generate_orbit(0.8, 1., n_points=100)).view(100, -1)
samples, e_hat, M_hat = infer_params(orbit)

print("true e, M:   ", 0.8, 1.)
print("pred mean e,M", e_hat, M_hat)


In [ ]:
cinn.eval()
encoder.eval()

def infer_params(orbit, n_samples_z=2000):
    with torch.no_grad():
        cond = ...         # [1, 2]
        cond = cond.repeat(n_samples_z, 1)         # [K, 2]
        z = ... # n_samples_z from N(0, I_2)
        params_samples, _ = cinn(z, c=[cond], rev=True)
    return params_samples.numpy()            # [K, 2]


In [ ]:
orbit = generate_orbit(0.8, 1., n_points = 100)
orbit = torch.tensor(orbit).view(100, -1)

samples = infer_params(orbit)

e_hat, M_hat  = ... 
print("true e, M:   ", 0.8, 1.)
print("pred mean e,M", e_hat, M_hat)


In [ ]:
def plot_posterior(idx=0, n_samples_z=2000):
    global orbits_t, params_t

    orbit = orbits_t[idx]
    true_e, true_M = params_t[idx].cpu().numpy()

    samples = infer_params(orbit, n_samples_z=n_samples_z)
    e_s = samples[:, 0]
    M_s = samples[:, 1]

    e_mean = e_s.mean()
    M_mean = M_s.mean()

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # e histogram
    axes[0].hist(e_s, bins=50, density=True, alpha=0.7)
    axes[0].axvline(e_mean, color='r', linestyle='--', label=f'mean={e_mean:.3f}')
    axes[0].axvline(true_e, color='k', linestyle=':', label=f'true={true_e:.3f}')
    axes[0].set_title('Posterior of e')
    axes[0].set_xlabel('e')
    axes[0].legend()

    # M histogram
    axes[1].hist(M_s, bins=50, density=True, alpha=0.7)
    axes[1].axvline(M_mean, color='r', linestyle='--', label=f'mean={M_mean:.3f}')
    axes[1].axvline(true_M, color='k', linestyle=':', label=f'true={true_M:.3f}')
    axes[1].set_title('Posterior of M')
    axes[1].set_xlabel('M')
    axes[1].legend()

    # joint scatter
    axes[2].scatter(e_s, M_s, s=5, alpha=0.25)
    axes[2].scatter([e_mean], [M_mean], color='r', s=80, marker='x', label='mean')
    axes[2].scatter([true_e], [true_M], color='k', s=80, marker='o', facecolors='none', label='true')
    axes[2].set_title('Joint posterior samples')
    axes[2].set_xlabel('e')
    axes[2].set_ylabel('M')
    axes[2].legend()

    plt.tight_layout()
    plt.show()

    print(f"true     : e={true_e:.4f}, M={true_M:.4f}")
    print(f"posterior: e_mean={e_mean:.4f}, M_mean={M_mean:.4f}")

plot_posterior(idx=239, n_samples_z=3000)